# HASTS 416/7 – Stochastic Processes | Group Work Project #1
## Volatility Modeling, Option Pricing & Interest Rate Simulation
---
**Underlying Asset:** SM Energy Company (SM)  
**Current Stock Price:** $232.90 USD  
**Risk-Free Rate:** 1.50% per annum  
**Trading Days per Year:** 250

In [ ]:
# ============================================================
# IMPORTS & GLOBAL SETUP
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize, differential_evolution
from scipy.interpolate import CubicSpline
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Global Parameters
S0    = 232.90
r     = 0.0150
T_days_year = 250

# Option Data
data = pd.DataFrame({
    'days': [15,15,15,15,15, 60,60,60,60,60, 120,120,120,120,120,
             15,15,15,15,15, 60,60,60,60,60, 120,120,120,120,120],
    'strike': [227.5,230,232.5,235,237.5]*6,
    'price':  [10.52,10.05,7.75,6.01,4.75,
                16.78,17.65,16.86,16.05,15.10,
                27.92,24.12,22.97,21.75,18.06,
                4.32,5.20,6.45,7.56,8.78,
                11.03,12.15,13.37,14.75,15.62,
                14.530629,16.25,17.22,18.74,19.73],
    'type':   ['C']*15 + ['P']*15
})
data['T'] = data['days'] / T_days_year

print('Option Data Loaded:')
print(data.to_string(index=False))

# ─── Vectorised integration helpers (shared by Lewis & Bates) ───────────────
# Integration grid: 400-point trapezoidal rule on [0, 100]
# This is ~100x faster than scipy.integrate.quad and avoids
# the 'float() on complex' TypeError that killed the original notebook.
_U  = np.linspace(1e-5, 100.0, 400)   # real part of integration variable
_du = _U[1] - _U[0]


---
## STEP 1 | Sub-group 1 (Members 1–3)
### Task (a): Heston (1993) Model Calibration — Lewis (2001) Approach
**Target Maturity:** 15 days  
**Pricing Method:** Lewis (2001) characteristic function approach  
**Error Function:** Mean Squared Error (MSE)

In [ ]:
# ============================================================
# QUESTION (a): HESTON MODEL — LEWIS (2001) APPROACH
# ============================================================

# FIX 1: Use the Albrecher et al. (2007) 'rotation trick' CF
#         which avoids branch-cut discontinuities for long maturities.
# FIX 2: Never cast phi to float — it can be complex (u − 0.5j for Lewis).
# FIX 3: Use np.trapezoid (NumPy ≥ 2.0) instead of deprecated np.trapz.

def heston_cf(phi, S0, T, r, kappa, theta, sigma, rho, v0):
    """Albrecher et al. (2007) stable Heston characteristic function.
    phi may be a real scalar, real array, or complex array."""
    i = 1j
    b  = kappa - rho * sigma * i * phi
    d  = np.sqrt((rho * sigma * i * phi - b)**2
                 - sigma**2 * (2 * i * phi - phi**2))
    g  = (b - d) / (b + d)
    eT = np.exp(-d * T)
    C  = (r * i * phi * T
          + (kappa * theta / sigma**2)
          * ((b - d) * T - 2 * np.log((1 - g * eT) / (1 - g))))
    D  = ((b - d) / sigma**2) * ((1 - eT) / (1 - g * eT))
    return np.exp(C + D * v0 + i * phi * np.log(S0))


def heston_lewis_call(S0, K, T, r, kappa, theta, sigma, rho, v0):
    """Lewis (2001) call price via numerical integration on Im(u)=−1/2 strip."""
    F   = S0 * np.exp(r * T)
    k   = np.log(K / F)                 # log-moneyness
    u_c = _U - 0.5j                     # complex integration variable
    cf  = heston_cf(u_c, S0, T, r, kappa, theta, sigma, rho, v0)
    integrand = np.real(
        np.exp(-1j * _U * k) * cf / (_U**2 + 0.25)
    )
    integral  = np.trapezoid(integrand, _U)
    call      = F * np.exp(-r * T) - (np.sqrt(K * F) * np.exp(-r * T) / np.pi) * integral
    return max(float(call), max(S0 - K * np.exp(-r * T), 0.0))


def heston_lewis_put(S0, K, T, r, kappa, theta, sigma, rho, v0):
    call = heston_lewis_call(S0, K, T, r, kappa, theta, sigma, rho, v0)
    return max(call - S0 + K * np.exp(-r * T), 0.0)


# ── MSE objective ──────────────────────────────────────────────────────────
def mse_lewis_15(params):
    kappa, theta, sigma, rho, v0 = params
    if kappa <= 0 or theta <= 0 or sigma <= 0 or v0 <= 0 or abs(rho) >= 1:
        return 1e10
    subset = data[data['days'] == 15]
    errs   = []
    for _, row in subset.iterrows():
        K, Tv, p = float(row['strike']), float(row['T']), float(row['price'])
        try:
            mp = (heston_lewis_call(S0, K, Tv, r, kappa, theta, sigma, rho, v0)
                  if row['type'] == 'C' else
                  heston_lewis_put (S0, K, Tv, r, kappa, theta, sigma, rho, v0))
            errs.append((mp - p) ** 2)
        except:
            return 1e10
    return float(np.mean(errs))


# ── Calibration ─────────────────────────────────────────────────────────────
# FIX 4: Correct parameter bounds.
#   theta and v0 are VARIANCES (not vols). For implied vols ~30-37%,
#   variances are ~0.09-0.14, so upper bound 0.50 is generous.
print('Calibrating Heston (Lewis) to 15-day options...')
bounds_heston = [
    (0.1,  15.0),   # kappa — mean-reversion speed
    (0.01,  0.50),  # theta — long-run VARIANCE  (vol 10%..71%)
    (0.01,  2.00),  # sigma — vol-of-vol
    (-0.99, 0.00),  # rho   — correlation (enforce leverage effect)
    (0.01,  0.50),  # v0    — initial VARIANCE
]
result_lewis = differential_evolution(
    mse_lewis_15, bounds_heston,
    seed=42, maxiter=500, tol=1e-9, popsize=20, mutation=(0.5, 1.5), recombination=0.9
)
kappa_l, theta_l, sigma_l, rho_l, v0_l = result_lewis.x

print(f'\nCalibrated Parameters (Lewis):')
print(f'  kappa (mean reversion speed) = {kappa_l:.4f}')
print(f'  theta (long-run variance)    = {theta_l:.6f}  => long-run vol = {np.sqrt(theta_l)*100:.2f}%')
print(f'  sigma (vol of vol)           = {sigma_l:.4f}')
print(f'  rho   (correlation)          = {rho_l:.4f}')
print(f'  v0    (initial variance)     = {v0_l:.6f}  => initial vol = {np.sqrt(v0_l)*100:.2f}%')
print(f'  MSE                          = {result_lewis.fun:.6f}')
print(f'  Feller condition (2κθ > σ²): {2*kappa_l*theta_l:.4f} > {sigma_l**2:.4f} => {2*kappa_l*theta_l > sigma_l**2}')


In [ ]:
# ── Plot Lewis calibration results ──────────────────────────────────────────
subset_15 = data[data['days'] == 15].copy()
model_prices_lewis = []
for _, row in subset_15.iterrows():
    K, Tv = float(row['strike']), float(row['T'])
    p = (heston_lewis_call(S0, K, Tv, r, kappa_l, theta_l, sigma_l, rho_l, v0_l)
         if row['type'] == 'C' else
         heston_lewis_put (S0, K, Tv, r, kappa_l, theta_l, sigma_l, rho_l, v0_l))
    model_prices_lewis.append(p)
subset_15['model_price'] = model_prices_lewis
subset_15['error']       = subset_15['model_price'] - subset_15['price']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for opt_type, color, marker in [('C', 'royalblue', 'o'), ('P', 'tomato', 's')]:
    sub   = subset_15[subset_15['type'] == opt_type]
    label = 'Call' if opt_type == 'C' else 'Put'
    axes[0].plot(sub['strike'], sub['price'],       marker=marker, linestyle='--', color=color,       label=f'{label} Market')
    axes[0].plot(sub['strike'], sub['model_price'], marker=marker, linestyle='-',  color=color, alpha=0.6, label=f'{label} Model')
axes[0].set_title('Heston Lewis (2001): Market vs Model Prices\n(15-day maturity)', fontsize=12)
axes[0].set_xlabel('Strike ($)'); axes[0].set_ylabel('Option Price ($)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].bar(range(len(subset_15)), subset_15['error'],
            color=['royalblue' if t == 'C' else 'tomato' for t in subset_15['type']])
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Pricing Errors (Model − Market)\nHeston Lewis (2001)', fontsize=12)
axes[1].set_xlabel('Option Index'); axes[1].set_ylabel('Error ($)')
axes[1].set_xticks(range(len(subset_15)))
axes[1].set_xticklabels([f"{r['type']}\nK={r['strike']}" for _, r in subset_15.iterrows()], fontsize=8)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('lewis_calibration_15d.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nDetailed Results:')
print(subset_15[['type','strike','price','model_price','error']].to_string(index=False))


---
## STEP 1 | Sub-group 2 (Members 4–6)
### Task (b): Heston (1993) Model Calibration — Carr-Madan (1999) Approach
**Target Maturity:** 15 days  
**Pricing Method:** Carr-Madan FFT approach  
**Error Function:** Mean Squared Error (MSE)

In [ ]:
# ============================================================
# QUESTION (b): HESTON MODEL — CARR-MADAN (1999) APPROACH
# ============================================================

# FIX: Use the same stable Albrecher CF (heston_cf) defined in Task (a).
#       The original used a different CF formulation which can diverge.

def carr_madan_call(S0, K, T, r, kappa, theta, sigma, rho, v0,
                    alpha=1.5, N=4096, eta=0.25):
    """Carr-Madan (1999) FFT-based call option pricing."""
    lam    = 2 * np.pi / (N * eta)
    b      = np.log(K) - N * lam / 2
    k_grid = b + lam * np.arange(N)
    v      = eta * np.arange(N)

    # Use same stable CF — NOTE: pass v-(alpha+1)*1j (complex!)
    cf  = heston_cf(v - (alpha + 1) * 1j, S0, T, r, kappa, theta, sigma, rho, v0)
    psi = (np.exp(-r * T) * cf
           / (alpha**2 + alpha - v**2 + 1j * (2 * alpha + 1) * v))

    w = eta * (3 + (-1)**np.arange(N) - (np.arange(N) == 0).astype(float)) / 3
    y = np.fft.fft(np.exp(1j * b * v) * psi * w)
    calls = (np.exp(-alpha * k_grid) / np.pi) * np.real(y)

    log_K = np.log(K)
    idx   = np.clip(np.searchsorted(k_grid, log_K), 1, N - 2)
    t_    = (log_K - k_grid[idx-1]) / (k_grid[idx] - k_grid[idx-1])
    return max(float(np.real(calls[idx-1] + t_ * (calls[idx] - calls[idx-1]))), 0.0)


def carr_madan_put(S0, K, T, r, kappa, theta, sigma, rho, v0):
    call = carr_madan_call(S0, K, T, r, kappa, theta, sigma, rho, v0)
    return max(call - S0 + K * np.exp(-r * T), 0.0)


def mse_cm_15(params):
    kappa, theta, sigma, rho, v0 = params
    if kappa <= 0 or theta <= 0 or sigma <= 0 or v0 <= 0 or abs(rho) >= 1:
        return 1e10
    subset = data[data['days'] == 15]
    errs   = []
    for _, row in subset.iterrows():
        K, Tv, p = float(row['strike']), float(row['T']), float(row['price'])
        try:
            mp = (carr_madan_call(S0, K, Tv, r, kappa, theta, sigma, rho, v0)
                  if row['type'] == 'C' else
                  carr_madan_put (S0, K, Tv, r, kappa, theta, sigma, rho, v0))
            errs.append((mp - p) ** 2)
        except:
            return 1e10
    return float(np.mean(errs))


print('Calibrating Heston (Carr-Madan) to 15-day options...')
result_cm = differential_evolution(
    mse_cm_15, bounds_heston,
    seed=42, maxiter=500, tol=1e-9, popsize=20, mutation=(0.5, 1.5), recombination=0.9
)
kappa_c, theta_c, sigma_c, rho_c, v0_c = result_cm.x

print(f'\nCalibrated Parameters (Carr-Madan):')
print(f'  kappa = {kappa_c:.4f}')
print(f'  theta = {theta_c:.6f}  => long-run vol = {np.sqrt(theta_c)*100:.2f}%')
print(f'  sigma = {sigma_c:.4f}')
print(f'  rho   = {rho_c:.4f}')
print(f'  v0    = {v0_c:.6f}  => initial vol = {np.sqrt(v0_c)*100:.2f}%')
print(f'  MSE   = {result_cm.fun:.6f}')
print(f'\nComparison Lewis vs Carr-Madan:')
for nm, vl, vc in zip(['kappa','theta','sigma','rho','v0'],
                       result_lewis.x, result_cm.x):
    print(f'  {nm}: Lewis={vl:.4f}, CM={vc:.4f}, diff={abs(vl-vc):.4f}')


In [ ]:
# ── Plot Carr-Madan calibration results ─────────────────────────────────────
subset_15 = data[data['days'] == 15].copy()
model_prices_cm = []
for _, row in subset_15.iterrows():
    K, Tv = float(row['strike']), float(row['T'])
    p = (carr_madan_call(S0, K, Tv, r, kappa_c, theta_c, sigma_c, rho_c, v0_c)
         if row['type'] == 'C' else
         carr_madan_put (S0, K, Tv, r, kappa_c, theta_c, sigma_c, rho_c, v0_c))
    model_prices_cm.append(p)
subset_15['model_price_cm']    = model_prices_cm
subset_15['model_price_lewis'] = model_prices_lewis
subset_15['error_cm']          = subset_15['model_price_cm']    - subset_15['price']
subset_15['error']             = subset_15['model_price_lewis'] - subset_15['price']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for opt_type, color, marker in [('C', 'royalblue', 'o'), ('P', 'tomato', 's')]:
    sub   = subset_15[subset_15['type'] == opt_type]
    label = 'Call' if opt_type == 'C' else 'Put'
    axes[0].plot(sub['strike'], sub['price'],             marker=marker, linestyle='--', color=color,       label=f'{label} Market')
    axes[0].plot(sub['strike'], sub['model_price_cm'],    marker=marker, linestyle='-',  color=color, alpha=0.6, label=f'{label} CM Model')
    axes[0].plot(sub['strike'], sub['model_price_lewis'], marker=marker, linestyle=':',  color='gray', alpha=0.6, label=f'{label} Lewis Model')
axes[0].set_title('Heston Carr-Madan (1999): Market vs Model Prices\n(15-day maturity)', fontsize=12)
axes[0].set_xlabel('Strike ($)'); axes[0].set_ylabel('Option Price ($)')
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)

x = np.arange(len(subset_15))
axes[1].bar(x - 0.2, subset_15['error'],    width=0.4, label='Lewis Error', color='steelblue', alpha=0.8)
axes[1].bar(x + 0.2, subset_15['error_cm'], width=0.4, label='CM Error',    color='coral',     alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Lewis vs Carr-Madan Pricing Errors\n(15-day maturity)', fontsize=12)
axes[1].set_xlabel('Option Index'); axes[1].set_ylabel('Error ($)')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{r['type']}K={r['strike']}" for _, r in subset_15.iterrows()], fontsize=7, rotation=45)
plt.tight_layout()
plt.savefig('cm_calibration_15d.png', dpi=150, bbox_inches='tight')
plt.show()


---
## STEP 1 | Sub-group 3 (Members 7–10)
### Task (c): Asian Call Option Pricing — Monte Carlo (20-day ATM)
**Instrument:** ATM Asian Call Option, K = S0 = $232.90  
**Maturity:** 20 trading days  
**Method:** Risk-neutral Monte Carlo under Heston dynamics  
**Bank Fee:** 4% markup on fair price

In [ ]:
# ============================================================
# QUESTION (c): ASIAN CALL OPTION PRICING — MONTE CARLO
# ============================================================

# FIX (the NameError): kappa_l etc. are defined in Task (a) above.
# You MUST run cells (a) before running this cell.
# The parameters below now reference the Lewis-calibrated values correctly.
kappa_use, theta_use, sigma_use, rho_use, v0_use = kappa_l, theta_l, sigma_l, rho_l, v0_l

K_asian      = S0          # ATM
T_asian_days = 20
T_asian      = T_asian_days / T_days_year
n_sims       = 100_000
np.random.seed(42)


def simulate_heston_paths(S0, v0, kappa, theta, sigma, rho, r, T, n_steps, n_sims):
    """Euler-Maruyama discretisation of Heston SDE.
    Returns stock paths: shape (n_sims, n_steps+1)."""
    dt = T / n_steps
    S  = np.zeros((n_sims, n_steps + 1))
    v  = np.zeros((n_sims, n_steps + 1))
    S[:, 0] = S0
    v[:, 0] = v0
    for t in range(n_steps):
        Z1 = np.random.standard_normal(n_sims)
        Z2 = np.random.standard_normal(n_sims)
        Zv = Z1
        Zs = rho * Z1 + np.sqrt(max(1 - rho**2, 0)) * Z2
        v_pos      = np.maximum(v[:, t], 0)
        v[:, t+1]  = np.maximum(
            v_pos + kappa * (theta - v_pos) * dt
            + sigma * np.sqrt(v_pos * dt) * Zv, 0)
        S[:, t+1]  = S[:, t] * np.exp(
            (r - 0.5 * v_pos) * dt + np.sqrt(v_pos * dt) * Zs)
    return S, v


print(f'Running {n_sims:,} Monte Carlo simulations for Asian Call...')
print(f'Parameters: K=${K_asian}, T={T_asian_days} days, S0=${S0}')

S_paths, v_paths = simulate_heston_paths(
    S0, v0_use, kappa_use, theta_use, sigma_use, rho_use,
    r, T_asian, T_asian_days, n_sims
)

avg_prices      = np.mean(S_paths, axis=1)
payoffs         = np.maximum(avg_prices - K_asian, 0)
discount_factor = np.exp(-r * T_asian)
fair_price      = discount_factor * np.mean(payoffs)
std_err         = discount_factor * np.std(payoffs) / np.sqrt(n_sims)
ci_lower        = fair_price - 1.96 * std_err
ci_upper        = fair_price + 1.96 * std_err
bank_fee        = 0.04
client_price    = fair_price * (1 + bank_fee)

print(f'\n=== ASIAN CALL OPTION RESULTS ===')
print(f'  Fair Price (Monte Carlo):    ${fair_price:.4f}')
print(f'  Standard Error:              ${std_err:.4f}')
print(f'  95% Confidence Interval:     [${ci_lower:.4f}, ${ci_upper:.4f}]')
print(f'  Bank Fee (4%):               ${fair_price * bank_fee:.4f}')
print(f'  Final Client Price:          ${client_price:.4f}')


In [ ]:
# ── Visualise Asian option simulation ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

n_plot = 200
t_grid = np.linspace(0, T_asian_days, T_asian_days + 1)
for i in range(n_plot):
    axes[0].plot(t_grid, S_paths[i], alpha=0.1, color='steelblue', linewidth=0.5)
axes[0].axhline(K_asian, color='red', linestyle='--', linewidth=1.5,
                label=f'Strike K=${K_asian:.0f}')
axes[0].set_title(f'Sample Heston Stock Paths (first {n_plot})', fontsize=11)
axes[0].set_xlabel('Trading Days'); axes[0].set_ylabel('Stock Price ($)')
axes[0].legend(); axes[0].grid(alpha=0.3)

itm = payoffs[payoffs > 0]
axes[1].hist(itm, bins=80, color='seagreen', edgecolor='white', alpha=0.8)
axes[1].axvline(np.mean(itm), color='red', linestyle='--',
                label=f'Mean payoff (ITM): ${np.mean(itm):.2f}')
axes[1].set_title('Distribution of In-the-Money Payoffs\n(Asian Call)', fontsize=11)
axes[1].set_xlabel('Payoff ($)'); axes[1].set_ylabel('Frequency')
axes[1].legend(); axes[1].grid(alpha=0.3)

batch_sizes = np.logspace(2, np.log10(n_sims), 100).astype(int)
conv_prices = [
    discount_factor * np.mean(np.maximum(np.mean(S_paths[:b], axis=1) - K_asian, 0))
    for b in batch_sizes
]
axes[2].semilogx(batch_sizes, conv_prices, color='darkorange')
axes[2].axhline(fair_price, color='red', linestyle='--',
                label=f'Final estimate: ${fair_price:.4f}')
axes[2].fill_between(batch_sizes, ci_lower, ci_upper,
                     alpha=0.2, color='red', label='95% CI')
axes[2].set_title('Monte Carlo Convergence\n(Asian Call Price)', fontsize=11)
axes[2].set_xlabel('Number of Simulations'); axes[2].set_ylabel('Estimated Price ($)')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('ATM Asian Call Option | 20 Days | Heston Dynamics', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('asian_call_mc.png', dpi=150, bbox_inches='tight')
plt.show()

pct_itm = 100 * np.mean(payoffs > 0)
print(f'\nSummary for client:')
print(f'  Fair value (theoretical): ${fair_price:.2f}')
print(f'  Price charged to client:  ${client_price:.2f} (inclusive of 4% service fee)')
print(f'  Probability of being ITM at expiry: {pct_itm:.1f}%')


---
## STEP 2 | Sub-group 3 (Members 7–10)
### Task (d): Bates (1996) Model Calibration — Lewis Approach (60-day maturity)

In [ ]:
# ============================================================
# QUESTION (d): BATES (1996) MODEL — LEWIS APPROACH — 60 DAYS
# ============================================================

# FIX: Use vectorised CF + trapezoid integration (same pattern as Task a).

def bates_cf(phi, S0, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j):
    """Bates (1996) = Heston + compound-Poisson (Merton) jumps."""
    i     = 1j
    k_bar = np.exp(mu_j + 0.5 * delta_j**2) - 1   # mean relative jump
    # Heston part (Albrecher rotation)
    b   = kappa - rho * sigma * i * phi
    d   = np.sqrt((rho * sigma * i * phi - b)**2
                  - sigma**2 * (2 * i * phi - phi**2))
    g   = (b - d) / (b + d)
    eT  = np.exp(-d * T)
    C   = ((r - lam_j * k_bar) * i * phi * T
           + (kappa * theta / sigma**2)
           * ((b - d) * T - 2 * np.log((1 - g * eT) / (1 - g))))
    D   = ((b - d) / sigma**2) * ((1 - eT) / (1 - g * eT))
    # Jump part
    Cj  = lam_j * T * (np.exp(i * phi * mu_j - 0.5 * delta_j**2 * phi**2) - 1 - i * phi * k_bar)
    return np.exp(C + D * v0 + i * phi * np.log(S0) + Cj)


def bates_lewis_call(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j):
    F         = S0 * np.exp(r * T)
    k         = np.log(K / F)
    u_c       = _U - 0.5j
    cf        = bates_cf(u_c, S0, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
    integrand = np.real(np.exp(-1j * _U * k) * cf / (_U**2 + 0.25))
    integral  = np.trapezoid(integrand, _U)
    call      = F * np.exp(-r * T) - (np.sqrt(K * F) * np.exp(-r * T) / np.pi) * integral
    return max(float(call), max(S0 - K * np.exp(-r * T), 0.0))


def bates_lewis_put(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j):
    call = bates_lewis_call(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
    return max(call - S0 + K * np.exp(-r * T), 0.0)


def mse_bates_lewis_60(params):
    kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j = params
    if (kappa<=0 or theta<=0 or sigma<=0 or v0<=0
            or abs(rho)>=1 or lam_j<0 or delta_j<=0):
        return 1e10
    subset = data[data['days'] == 60]
    errs   = []
    for _, row in subset.iterrows():
        K, Tv, p = float(row['strike']), float(row['T']), float(row['price'])
        try:
            mp = (bates_lewis_call(S0, K, Tv, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
                  if row['type'] == 'C' else
                  bates_lewis_put (S0, K, Tv, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j))
            errs.append((mp - p) ** 2)
        except:
            return 1e10
    return float(np.mean(errs))


print('Calibrating Bates (Lewis) to 60-day options...')
bounds_bates = [
    (0.1,  15.0),   # kappa
    (0.01,  0.50),  # theta (variance)
    (0.01,  2.00),  # sigma
    (-0.99, 0.00),  # rho
    (0.01,  0.50),  # v0 (variance)
    (0.0,   5.00),  # lambda_j  (jump intensity, jumps/year)
    (-0.50, 0.50),  # mu_j      (mean log jump size)
    (0.001, 0.50),  # delta_j   (jump vol)
]
result_bates_l = differential_evolution(
    mse_bates_lewis_60, bounds_bates,
    seed=42, maxiter=500, tol=1e-9, popsize=20, mutation=(0.5,1.5), recombination=0.9
)
kappa_bl, theta_bl, sigma_bl, rho_bl, v0_bl, lam_bl, mu_bl, delta_bl = result_bates_l.x

print(f'\nBates (Lewis) Calibrated Parameters — 60 days:')
print(f'  kappa  = {kappa_bl:.4f}')
print(f'  theta  = {theta_bl:.6f}  (long-run vol = {np.sqrt(theta_bl)*100:.2f}%)')
print(f'  sigma  = {sigma_bl:.4f}')
print(f'  rho    = {rho_bl:.4f}')
print(f'  v0     = {v0_bl:.6f}  (initial vol = {np.sqrt(v0_bl)*100:.2f}%)')
print(f'  lambda = {lam_bl:.4f}  (jumps/year)')
print(f'  mu_j   = {mu_bl:.4f}  (mean log jump)')
print(f'  delta  = {delta_bl:.4f}  (jump vol)')
print(f'  MSE    = {result_bates_l.fun:.6f}')


In [ ]:
# ── Plot Bates (Lewis) calibration ──────────────────────────────────────────
subset_60 = data[data['days'] == 60].copy()
bates_prices_l = []
for _, row in subset_60.iterrows():
    K, Tv = float(row['strike']), float(row['T'])
    p = (bates_lewis_call(S0, K, Tv, r, kappa_bl, theta_bl, sigma_bl, rho_bl, v0_bl, lam_bl, mu_bl, delta_bl)
         if row['type'] == 'C' else
         bates_lewis_put (S0, K, Tv, r, kappa_bl, theta_bl, sigma_bl, rho_bl, v0_bl, lam_bl, mu_bl, delta_bl))
    bates_prices_l.append(p)
subset_60['model_bates_l'] = bates_prices_l
subset_60['error_bates_l'] = subset_60['model_bates_l'] - subset_60['price']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for opt_type, color, marker in [('C','royalblue','o'),('P','tomato','s')]:
    sub   = subset_60[subset_60['type'] == opt_type]
    label = 'Call' if opt_type == 'C' else 'Put'
    axes[0].plot(sub['strike'], sub['price'],         marker=marker, linestyle='--', color=color, label=f'{label} Market')
    axes[0].plot(sub['strike'], sub['model_bates_l'], marker=marker, linestyle='-',  color=color, alpha=0.6, label=f'{label} Bates-Lewis')
axes[0].set_title('Bates (1996) Lewis Approach: Market vs Model\n(60-day maturity)', fontsize=12)
axes[0].set_xlabel('Strike ($)'); axes[0].set_ylabel('Option Price ($)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].bar(range(len(subset_60)), subset_60['error_bates_l'],
            color=['royalblue' if t == 'C' else 'tomato' for t in subset_60['type']])
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Pricing Errors — Bates Lewis (60 days)', fontsize=12)
axes[1].set_xlabel('Option'); axes[1].set_ylabel('Error ($)')
axes[1].set_xticks(range(len(subset_60)))
axes[1].set_xticklabels([f"{r['type']}K={r['strike']}" for _, r in subset_60.iterrows()], fontsize=8, rotation=45)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('bates_lewis_60d.png', dpi=150, bbox_inches='tight')
plt.show()


---
## STEP 2 | Sub-group 1 (Members 1–3)
### Task (e): Bates (1996) Model Calibration — Carr-Madan Approach (60-day maturity)

In [ ]:
# ============================================================
# QUESTION (e): BATES (1996) — CARR-MADAN APPROACH — 60 DAYS
# ============================================================

def bates_cm_call(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j,
                  alpha=1.5, N=4096, eta=0.25):
    """Carr-Madan FFT pricing with Bates characteristic function."""
    lam    = 2 * np.pi / (N * eta)
    b      = np.log(K) - N * lam / 2
    k_grid = b + lam * np.arange(N)
    v      = eta * np.arange(N)
    cf  = bates_cf(v - (alpha + 1) * 1j, S0, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
    psi = np.exp(-r * T) * cf / (alpha**2 + alpha - v**2 + 1j * (2 * alpha + 1) * v)
    w   = eta * (3 + (-1)**np.arange(N) - (np.arange(N) == 0).astype(float)) / 3
    y   = np.fft.fft(np.exp(1j * b * v) * psi * w)
    calls   = (np.exp(-alpha * k_grid) / np.pi) * np.real(y)
    log_K   = np.log(K)
    idx     = np.clip(np.searchsorted(k_grid, log_K), 1, N - 2)
    t_      = (log_K - k_grid[idx-1]) / (k_grid[idx] - k_grid[idx-1])
    return max(float(np.real(calls[idx-1] + t_ * (calls[idx] - calls[idx-1]))), 0.0)


def bates_cm_put(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j):
    call = bates_cm_call(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
    return max(call - S0 + K * np.exp(-r * T), 0.0)


def mse_bates_cm_60(params):
    kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j = params
    if (kappa<=0 or theta<=0 or sigma<=0 or v0<=0
            or abs(rho)>=1 or lam_j<0 or delta_j<=0):
        return 1e10
    subset = data[data['days'] == 60]
    errs   = []
    for _, row in subset.iterrows():
        K, Tv, p = float(row['strike']), float(row['T']), float(row['price'])
        try:
            mp = (bates_cm_call(S0, K, Tv, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
                  if row['type'] == 'C' else
                  bates_cm_put (S0, K, Tv, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j))
            errs.append((mp - p) ** 2)
        except:
            return 1e10
    return float(np.mean(errs))


print('Calibrating Bates (Carr-Madan) to 60-day options...')
result_bates_cm = differential_evolution(
    mse_bates_cm_60, bounds_bates,
    seed=42, maxiter=500, tol=1e-9, popsize=20, mutation=(0.5,1.5), recombination=0.9
)
kappa_bc, theta_bc, sigma_bc, rho_bc, v0_bc, lam_bc, mu_bc, delta_bc = result_bates_cm.x

print(f'\nBates (Carr-Madan) Calibrated Parameters — 60 days:')
print(f'  kappa  = {kappa_bc:.4f}')
print(f'  theta  = {theta_bc:.6f}  (long-run vol = {np.sqrt(theta_bc)*100:.2f}%)')
print(f'  sigma  = {sigma_bc:.4f}')
print(f'  rho    = {rho_bc:.4f}')
print(f'  v0     = {v0_bc:.6f}  (initial vol = {np.sqrt(v0_bc)*100:.2f}%)')
print(f'  lambda = {lam_bc:.4f}  (jumps/year)')
print(f'  mu_j   = {mu_bc:.4f}')
print(f'  delta  = {delta_bc:.4f}')
print(f'  MSE    = {result_bates_cm.fun:.6f}')
print(f'\nComparison Bates Lewis vs Carr-Madan (60d):')
names = ['kappa','theta','sigma','rho','v0','lambda','mu_j','delta']
for nm, bl, bc in zip(names, result_bates_l.x, result_bates_cm.x):
    print(f'  {nm:6s}: Lewis={bl:.4f}, CM={bc:.4f}')


In [ ]:
# ── Plot comparison Bates Lewis vs Carr-Madan ───────────────────────────────
bates_prices_cm = []
for _, row in subset_60.iterrows():
    K, Tv = float(row['strike']), float(row['T'])
    p = (bates_cm_call(S0, K, Tv, r, kappa_bc, theta_bc, sigma_bc, rho_bc, v0_bc, lam_bc, mu_bc, delta_bc)
         if row['type'] == 'C' else
         bates_cm_put (S0, K, Tv, r, kappa_bc, theta_bc, sigma_bc, rho_bc, v0_bc, lam_bc, mu_bc, delta_bc))
    bates_prices_cm.append(p)
subset_60['model_bates_cm'] = bates_prices_cm
subset_60['error_bates_cm'] = subset_60['model_bates_cm'] - subset_60['price']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for opt_type, color, marker in [('C','royalblue','o'),('P','tomato','s')]:
    sub   = subset_60[subset_60['type'] == opt_type]
    label = 'Call' if opt_type == 'C' else 'Put'
    axes[0].plot(sub['strike'], sub['price'],           marker=marker, linestyle='--', color=color,       label=f'{label} Market')
    axes[0].plot(sub['strike'], sub['model_bates_l'],   marker=marker, linestyle='-',  color=color, alpha=0.7, label=f'{label} Lewis')
    axes[0].plot(sub['strike'], sub['model_bates_cm'],  marker=marker, linestyle=':',  color=color, alpha=0.7, label=f'{label} CM')
axes[0].set_title('Bates 60d: Market vs Lewis vs Carr-Madan', fontsize=11)
axes[0].set_xlabel('Strike ($)'); axes[0].set_ylabel('Price ($)')
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)

x = np.arange(len(subset_60))
axes[1].bar(x - 0.2, subset_60['error_bates_l'],  width=0.4, label='Lewis Error', color='steelblue', alpha=0.8)
axes[1].bar(x + 0.2, subset_60['error_bates_cm'], width=0.4, label='CM Error',    color='coral',     alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Bates 60d: Pricing Errors Comparison', fontsize=11)
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{r['type']}K={r['strike']}" for _, r in subset_60.iterrows()], fontsize=7, rotation=45)
plt.tight_layout()
plt.savefig('bates_cm_60d.png', dpi=150, bbox_inches='tight')
plt.show()


---
## STEP 2 | Sub-group 2 (Members 4–6)
### Task (f): Put Option Pricing — Monte Carlo (70 days, 95% Moneyness)

In [ ]:
# ============================================================
# QUESTION (f): PUT OPTION — MONTE CARLO — 70 DAYS, 95% MONEYNESS
# ============================================================

K_put      = 0.95 * S0          # 95% moneyness = OTM put
T_put_days = 70
T_put      = T_put_days / T_days_year
n_sims_put = 100_000
np.random.seed(123)

print(f'Put Option Parameters:')
print(f'  S0 = ${S0}, K = ${K_put:.4f} (95% moneyness), T = {T_put_days} days')


def simulate_bates_paths(S0, v0, kappa, theta, sigma, rho, r,
                          lam_j, mu_j, delta_j, T, n_steps, n_sims):
    """Euler-Maruyama for Bates model (Heston + Merton jumps)."""
    dt    = T / n_steps
    k_bar = np.exp(mu_j + 0.5 * delta_j**2) - 1
    S     = np.zeros((n_sims, n_steps + 1))
    v     = np.zeros((n_sims, n_steps + 1))
    S[:, 0] = S0
    v[:, 0] = v0
    rng = np.random.default_rng(seed=123)
    for t in range(n_steps):
        Z1     = np.random.standard_normal(n_sims)
        Z2     = np.random.standard_normal(n_sims)
        Zv     = Z1
        Zs     = rho * Z1 + np.sqrt(max(1 - rho**2, 0)) * Z2
        v_pos  = np.maximum(v[:, t], 0)
        # Jump component (vectorised Poisson sampling)
        n_jmp  = np.random.poisson(lam_j * dt, n_sims)
        j_size = np.array([np.sum(np.random.normal(mu_j, delta_j, nj))
                            if nj > 0 else 0.0 for nj in n_jmp])
        v[:, t+1] = np.maximum(
            v_pos + kappa * (theta - v_pos) * dt
            + sigma * np.sqrt(v_pos * dt) * Zv, 0)
        S[:, t+1] = (S[:, t]
                     * np.exp((r - lam_j * k_bar - 0.5 * v_pos) * dt
                              + np.sqrt(v_pos * dt) * Zs)
                     * np.exp(j_size))
    return S, v


print(f'\nRunning {n_sims_put:,} simulations for Put option (Bates model)...')
S_put, _ = simulate_bates_paths(
    S0, v0_bl, kappa_bl, theta_bl, sigma_bl, rho_bl, r,
    lam_bl, mu_bl, delta_bl, T_put, T_put_days, n_sims_put
)

put_payoffs    = np.maximum(K_put - S_put[:, -1], 0)
put_fair_price = np.exp(-r * T_put) * np.mean(put_payoffs)
put_std_err    = np.exp(-r * T_put) * np.std(put_payoffs) / np.sqrt(n_sims_put)
put_ci         = (put_fair_price - 1.96 * put_std_err,
                  put_fair_price + 1.96 * put_std_err)

print(f'\n=== PUT OPTION RESULTS ===')
print(f'  Strike (95% moneyness):      ${K_put:.4f}')
print(f'  Fair Price (Monte Carlo):    ${put_fair_price:.4f}')
print(f'  Standard Error:              ${put_std_err:.4f}')
print(f'  95% CI:                      [${put_ci[0]:.4f}, ${put_ci[1]:.4f}]')
print(f'  Probability ITM:             {100*np.mean(put_payoffs>0):.1f}%')


In [ ]:
# ── Put option plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

t_grid = np.linspace(0, T_put_days, T_put_days + 1)
for i in range(200):
    axes[0].plot(t_grid, S_put[i], alpha=0.1, color='steelblue', linewidth=0.5)
axes[0].axhline(K_put, color='red',   linestyle='--', linewidth=1.5, label=f'Strike K=${K_put:.0f}')
axes[0].axhline(S0,    color='green', linestyle=':',  linewidth=1.5, label=f'S0=${S0}')
axes[0].set_title(f'Bates Stock Paths (first 200)\nPut Option Pricing', fontsize=11)
axes[0].set_xlabel('Trading Days'); axes[0].set_ylabel('Price ($)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].hist(S_put[:, -1], bins=100, color='steelblue', edgecolor='white', alpha=0.8, density=True)
axes[1].axvline(K_put, color='red',   linestyle='--', label=f'Strike K=${K_put:.0f}')
axes[1].axvline(S0,    color='green', linestyle=':',  label=f'S0=${S0}')
axes[1].set_title('Distribution of Final Stock Prices\n(70-day Bates)', fontsize=11)
axes[1].set_xlabel('S_T ($)'); axes[1].set_ylabel('Density')
axes[1].legend(); axes[1].grid(alpha=0.3)

batch = np.logspace(2, np.log10(n_sims_put), 100).astype(int)
conv  = [np.exp(-r * T_put) * np.mean(np.maximum(K_put - S_put[:b, -1], 0)) for b in batch]
axes[2].semilogx(batch, conv, color='darkorange')
axes[2].axhline(put_fair_price, color='red', linestyle='--', label=f'Estimate: ${put_fair_price:.4f}')
axes[2].set_title('MC Convergence — Put Option', fontsize=11)
axes[2].set_xlabel('Simulations'); axes[2].set_ylabel('Estimated Price ($)')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle(f'SM Put Option | K=${K_put:.2f} (95%) | 70 Days | Bates Model', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('put_option_mc.png', dpi=150, bbox_inches='tight')
plt.show()


---
## STEP 3 | Full Team
### Task (g): CIR (1985) Model Calibration — Euribor Term Structure

In [ ]:
# ============================================================
# QUESTION (g): CIR MODEL CALIBRATION — EURIBOR
# ============================================================

euribor_tenors_days = np.array([7, 30, 90, 180, 360])
euribor_tenors_yr   = euribor_tenors_days / 365.0
euribor_rates       = np.array([0.648, 0.679, 1.173, 1.809, 2.556]) / 100

print('Market Euribor Term Structure:')
for tenor, rate in zip(['1W','1M','3M','6M','12M'], euribor_rates):
    print(f'  Euribor {tenor}: {rate*100:.3f}%')

# Cubic spline interpolation to weekly grid
cs       = CubicSpline(euribor_tenors_yr, euribor_rates, bc_type='not-a-knot')
t_weekly = np.arange(1, 53) / 52.0
r_weekly = np.maximum(cs(t_weekly), 0.001)


def cir_bond_price(T, r0, kappa, theta, sigma):
    """CIR (1985) zero-coupon bond price P(0,T)."""
    gamma = np.sqrt(kappa**2 + 2 * sigma**2)
    A_num = 2 * gamma * np.exp((kappa + gamma) * T / 2)
    A_den = (kappa + gamma) * (np.exp(gamma * T) - 1) + 2 * gamma
    A     = (A_num / A_den) ** (2 * kappa * theta / sigma**2)
    B     = 2 * (np.exp(gamma * T) - 1) / A_den
    return A * np.exp(-B * r0)


def cir_yield(T, r0, kappa, theta, sigma):
    return -np.log(cir_bond_price(T, r0, kappa, theta, sigma)) / T


r0_cir = float(euribor_rates[0])


def mse_cir(params):
    kappa, theta, sigma = params
    if kappa <= 0 or theta <= 0 or sigma <= 0:
        return 1e10
    if 2 * kappa * theta <= sigma**2:        # Feller condition
        return 1e10
    errs = [(cir_yield(float(t), r0_cir, kappa, theta, sigma) - float(rm))**2
            for t, rm in zip(t_weekly, r_weekly)]
    return float(np.mean(errs))


print('\nCalibrating CIR model to Euribor term structure...')
result_cir = differential_evolution(
    mse_cir, [(0.01, 5.0), (0.001, 0.20), (0.001, 0.50)],
    seed=42, maxiter=1000, tol=1e-12, popsize=20
)
kappa_cir, theta_cir, sigma_cir = result_cir.x

print(f'\nCIR Calibrated Parameters:')
print(f'  kappa (speed of mean reversion) = {kappa_cir:.4f}')
print(f'  theta (long-run rate)           = {theta_cir*100:.4f}%')
print(f'  sigma (vol of short rate)       = {sigma_cir:.4f}')
print(f'  r0    (initial rate)            = {r0_cir*100:.4f}%')
print(f'  MSE                             = {result_cir.fun:.2e}')
print(f'  Feller (2κθ > σ²):              {2*kappa_cir*theta_cir:.4f} > {sigma_cir**2:.4f} => {2*kappa_cir*theta_cir > sigma_cir**2}')

print('\nFit at market tenors:')
for tenor, T, rm in zip(['1W','1M','3M','6M','12M'], euribor_tenors_yr, euribor_rates):
    rm_mod = cir_yield(float(T), r0_cir, kappa_cir, theta_cir, sigma_cir)
    print(f'  {tenor}: Market={rm*100:.4f}%  CIR={rm_mod*100:.4f}%  Error={abs(rm_mod-rm)*10000:.2f}bps')


In [ ]:
# ── CIR fit plot ─────────────────────────────────────────────────────────────
t_fine       = np.linspace(1/365, 1.0, 200)
r_cir_fit    = np.array([cir_yield(float(t), r0_cir, kappa_cir, theta_cir, sigma_cir) for t in t_fine])
r_weekly_fit = np.array([cir_yield(float(t), r0_cir, kappa_cir, theta_cir, sigma_cir) for t in t_weekly])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(t_fine*12, r_cir_fit*100, color='royalblue', linewidth=2, label='CIR Model Fit')
axes[0].plot(t_weekly*12, r_weekly*100, color='gray', linestyle='--', linewidth=1.2, label='Cubic Spline (Market)')
axes[0].scatter(euribor_tenors_yr*12, euribor_rates*100, color='red', zorder=5, s=80, label='Market Euribor')
axes[0].set_title('CIR Model vs Market Term Structure\n(Euribor)', fontsize=12)
axes[0].set_xlabel('Maturity (months)'); axes[0].set_ylabel('Rate (%)')
axes[0].legend(); axes[0].grid(alpha=0.3)

residuals = (r_weekly_fit - r_weekly) * 100
axes[1].bar(range(len(t_weekly)), residuals, color='steelblue', alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('CIR Fitting Residuals (Model − Market)\n(in % of rate)', fontsize=12)
axes[1].set_xlabel('Week'); axes[1].set_ylabel('Residual (% rate)')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('cir_calibration.png', dpi=150, bbox_inches='tight')
plt.show()


---
## STEP 3 | Full Team
### Task (h): CIR — Monte Carlo Simulation of 12-Month Euribor (100,000 paths)

In [ ]:
# ============================================================
# QUESTION (h): CIR MONTE CARLO — 12-MONTH EURIBOR SIMULATION
# ============================================================

n_sims_cir = 100_000
n_days_cir = 365
dt_cir     = 1.0 / 365
np.random.seed(42)

print(f'Running {n_sims_cir:,} CIR simulations ({n_days_cir} daily steps)...')

r_cir = np.full((n_sims_cir, n_days_cir + 1), r0_cir)
for t in range(n_days_cir):
    Z     = np.random.standard_normal(n_sims_cir)
    r_pos = np.maximum(r_cir[:, t], 0)
    dr    = (kappa_cir * (theta_cir - r_pos) * dt_cir
             + sigma_cir * np.sqrt(r_pos * dt_cir) * Z)
    r_cir[:, t+1] = np.maximum(r_cir[:, t] + dr, 0)

r_terminal           = r_cir[:, -1]
r_exp                = np.mean(r_terminal)
r_low                = np.quantile(r_terminal, 0.025)
r_high               = np.quantile(r_terminal, 0.975)
current_12m_euribor  = float(euribor_rates[-1])

print(f'\n=== CIR SIMULATION RESULTS (12M Euribor in 1 Year) ===')
print(f'  Expected Value (1 year):     {r_exp*100:.4f}%')
print(f'  95% Confidence Interval:     [{r_low*100:.4f}%, {r_high*100:.4f}%]')
print(f'  Current 12M Euribor:         {current_12m_euribor*100:.4f}%')
print(f'  Expected Change:             {(r_exp-current_12m_euribor)*100:+.4f}% ({(r_exp-current_12m_euribor)/current_12m_euribor*100:.1f}% relative)')
print(f'  P(rate rises):               {np.mean(r_terminal > current_12m_euribor)*100:.1f}%')
print(f'  P(rate falls):               {np.mean(r_terminal < current_12m_euribor)*100:.1f}%')


In [ ]:
# ── CIR Monte Carlo Plots ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

t_plot = np.linspace(0, 365, n_days_cir + 1)
for i in range(500):
    axes[0].plot(t_plot, r_cir[i]*100, alpha=0.03, color='steelblue', linewidth=0.5)
axes[0].plot(t_plot, np.mean(r_cir, axis=0)*100, color='red', linewidth=2, label='Mean path')
axes[0].plot(t_plot, np.quantile(r_cir, 0.025, axis=0)*100, color='orange', linewidth=1.5, linestyle='--', label='95% CI')
axes[0].plot(t_plot, np.quantile(r_cir, 0.975, axis=0)*100, color='orange', linewidth=1.5, linestyle='--')
axes[0].set_title(f'CIR Rate Paths ({n_sims_cir:,} sims)', fontsize=11)
axes[0].set_xlabel('Days'); axes[0].set_ylabel('Rate (%)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].hist(r_terminal*100, bins=150, color='steelblue', edgecolor='white', alpha=0.8, density=True)
axes[1].axvline(r_exp*100,               color='red',    linestyle='-',  linewidth=2,   label=f'E[r] = {r_exp*100:.3f}%')
axes[1].axvline(r_low*100,               color='orange', linestyle='--', linewidth=1.5, label=f'95% CI [{r_low*100:.3f}%, {r_high*100:.3f}%]')
axes[1].axvline(r_high*100,              color='orange', linestyle='--', linewidth=1.5)
axes[1].axvline(current_12m_euribor*100, color='green',  linestyle=':',  linewidth=2,   label=f'Current 12M: {current_12m_euribor*100:.3f}%')
axes[1].set_title('Distribution of 12M Euribor in 1 Year\n(CIR Monte Carlo)', fontsize=11)
axes[1].set_xlabel('Rate (%)'); axes[1].set_ylabel('Density')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

for pct, c in zip([5,25,50,75,95], ['#d62728','#ff7f0e','#2ca02c','#1f77b4','#9467bd']):
    axes[2].plot(t_plot, np.quantile(r_cir, pct/100, axis=0)*100, label=f'P{pct}', color=c, linewidth=1.5)
axes[2].set_title('CIR Percentile Bands (1 Year)', fontsize=11)
axes[2].set_xlabel('Days'); axes[2].set_ylabel('Rate (%)')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('CIR (1985) — Euribor 12-Month Rate Simulation', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('cir_mc_simulation.png', dpi=150, bbox_inches='tight')
plt.show()


---
## SUMMARY OF ALL RESULTS

In [ ]:
# ============================================================
# SUMMARY TABLE
# ============================================================
print('='*65)
print('HASTS 416/7 — GWP1 COMPLETE RESULTS SUMMARY')
print('='*65)
print(f'\n[Step 1a] Heston Lewis (15d):')
print(f'  κ={kappa_l:.4f}  θ={theta_l:.4f} ({np.sqrt(theta_l)*100:.2f}% vol)  σ={sigma_l:.4f}  ρ={rho_l:.4f}  v0={v0_l:.4f}  MSE={result_lewis.fun:.4f}')
print(f'[Step 1b] Heston CM (15d):')
print(f'  κ={kappa_c:.4f}  θ={theta_c:.4f} ({np.sqrt(theta_c)*100:.2f}% vol)  σ={sigma_c:.4f}  ρ={rho_c:.4f}  v0={v0_c:.4f}  MSE={result_cm.fun:.4f}')
print(f'[Step 1c] ATM Asian Call (20d): Fair=${fair_price:.4f}  Client=${client_price:.4f}  95%CI=[${ci_lower:.4f}, ${ci_upper:.4f}]')
print(f'[Step 2d] Bates Lewis (60d):   κ={kappa_bl:.4f}  θ={theta_bl:.4f}  σ={sigma_bl:.4f}  ρ={rho_bl:.4f}')
print(f'          Jump: λ={lam_bl:.4f}  μ={mu_bl:.4f}  δ={delta_bl:.4f}  MSE={result_bates_l.fun:.4f}')
print(f'[Step 2e] Bates CM (60d):      κ={kappa_bc:.4f}  θ={theta_bc:.4f}  σ={sigma_bc:.4f}  ρ={rho_bc:.4f}')
print(f'          Jump: λ={lam_bc:.4f}  μ={mu_bc:.4f}  δ={delta_bc:.4f}  MSE={result_bates_cm.fun:.4f}')
print(f'[Step 2f] Put Option (70d, K=95%): Fair=${put_fair_price:.4f}  95%CI=[${put_ci[0]:.4f}, ${put_ci[1]:.4f}]  P(ITM)={100*np.mean(put_payoffs>0):.1f}%')
print(f'[Step 3g] CIR Calibration:     κ={kappa_cir:.4f}  θ={theta_cir*100:.4f}%  σ={sigma_cir:.4f}  MSE={result_cir.fun:.2e}')
print(f'[Step 3h] 12M Euribor in 1yr:  E[r]={r_exp*100:.4f}%  95%CI=[{r_low*100:.4f}%, {r_high*100:.4f}%]')
print('='*65)
